# OCR Detection Verification

This notebook verifies that text detections are being properly logged and accumulated from the urbanocr pipeline.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path

# Configuration
PARQUET_PATH = "/share/pierson/matt/mllmsci/outputs/ocr/detections.parquet"


## 1. Load and Inspect Data


In [ ]:
df = pd.read_parquet(PARQUET_PATH)
print(f"Total detection rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
df.head(10)


## 2. Detection Count Statistics


In [ ]:
# Count detections per image
detections_per_image = df.groupby('image_path').size().sort_values(ascending=False)

print("=== DETECTION STATISTICS ===")
print(f"Unique images processed: {len(detections_per_image):,}")
print(f"Total detections: {len(df):,}")
print(f"\nDetections per image:")
print(f"  Max: {detections_per_image.max()}")
print(f"  Min: {detections_per_image.min()}")
print(f"  Mean: {detections_per_image.mean():.2f}")
print(f"  Median: {detections_per_image.median():.1f}")
print(f"  Std: {detections_per_image.std():.2f}")


In [ ]:
# Distribution of detection counts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(detections_per_image.values, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Detections per Image')
axes[0].set_ylabel('Number of Images')
axes[0].set_title('Distribution of Text Detections per Image')
axes[0].axvline(detections_per_image.mean(), color='red', linestyle='--', label=f'Mean: {detections_per_image.mean():.1f}')
axes[0].legend()

# Bar chart for images with most detections
top_10 = detections_per_image.head(10)
short_names = [Path(p).parent.parent.name + '/' + Path(p).name for p in top_10.index]
axes[1].barh(range(len(top_10)), top_10.values)
axes[1].set_yticks(range(len(top_10)))
axes[1].set_yticklabels(short_names, fontsize=8)
axes[1].set_xlabel('Number of Detections')
axes[1].set_title('Top 10 Images by Detection Count')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


## 3. Images with Multiple Detections (Proof of Accumulation)


In [ ]:
# Show images with many detections
multi_detection_images = detections_per_image[detections_per_image > 10]
print(f"Images with >10 detections: {len(multi_detection_images):,}")
print(f"Images with >20 detections: {len(detections_per_image[detections_per_image > 20]):,}")
print(f"Images with >50 detections: {len(detections_per_image[detections_per_image > 50]):,}")


In [ ]:
# Analyze and remove duplicates
print("=== DUPLICATE DETECTION ANALYSIS ===")

# Define dedup columns
dedup_cols = ["image_path", "text", "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2"]

# Count before dedup
rows_before = len(df)
has_text = df["text"].notna()
df_deduped = pd.concat([
    df[has_text].drop_duplicates(subset=dedup_cols, keep="first"),
    df[~has_text]
], ignore_index=True)
rows_after = len(df_deduped)

print(f"Rows before deduplication: {rows_before:,}")
print(f"Rows after deduplication: {rows_after:,}")
print(f"Duplicates removed: {rows_before - rows_after:,} ({100*(rows_before-rows_after)/rows_before:.1f}%)")

# Show most duplicated detections
print("\n=== MOST DUPLICATED DETECTIONS ===")
dup_counts = df[has_text].groupby(dedup_cols).size().reset_index(name='count')
worst_dups = dup_counts[dup_counts['count'] > 5].sort_values('count', ascending=False).head(10)
for _, row in worst_dups.iterrows():
    img_short = Path(row['image_path']).parent.parent.name + '/' + Path(row['image_path']).name
    print(f"  '{row['text']}' in {img_short}: {row['count']} copies")


In [ ]:
# Examine a high-detection image
top_image_path = detections_per_image.index[0]
top_image_detections = df[df['image_path'] == top_image_path]

print(f"=== EXAMPLE: Image with {len(top_image_detections)} detections ===")
print(f"Image: {top_image_path}")
print(f"\nAll detected texts:")
for i, row in top_image_detections.iterrows():
    print(f"  [{row['text_type']}] '{row['text']}' @ ({row['bbox_x1']}, {row['bbox_y1']}) - ({row['bbox_x2']}, {row['bbox_y2']})")


## 4. Visualize Detections on Image


In [ ]:
def visualize_detections(image_path, detections_df, max_detections=50):
    """Visualize OCR detections on an image."""
    # Load image
    img = Image.open(image_path)
    img_width, img_height = img.size
    
    # Get coordinate range from data
    if len(detections_df) > 0:
        bbox_max_x = detections_df['bbox_max_x'].iloc[0]
        bbox_max_y = detections_df['bbox_max_y'].iloc[0]
    else:
        bbox_max_x = bbox_max_y = 999
    
    fig, ax = plt.subplots(1, 1, figsize=(16, 16))
    ax.imshow(img)
    
    # Color map for text types
    colors = plt.cm.tab10.colors
    text_types = detections_df['text_type'].unique()
    color_map = {t: colors[i % len(colors)] for i, t in enumerate(text_types)}
    
    for i, (_, row) in enumerate(detections_df.iterrows()):
        if i >= max_detections:
            break
            
        # Convert global coords to pixel coords
        x1 = row['bbox_x1'] / bbox_max_x * img_width
        y1 = row['bbox_y1'] / bbox_max_y * img_height
        x2 = row['bbox_x2'] / bbox_max_x * img_width
        y2 = row['bbox_y2'] / bbox_max_y * img_height
        
        width = x2 - x1
        height = y2 - y1
        
        color = color_map.get(row['text_type'], 'red')
        rect = patches.Rectangle((x1, y1), width, height, 
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        
        # Add text label
        label = row['text'][:20] + '...' if len(str(row['text'])) > 20 else row['text']
        ax.text(x1, y1-5, label, fontsize=6, color=color, 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    ax.set_title(f"{Path(image_path).name} - {len(detections_df)} detections")
    ax.axis('off')
    
    # Legend
    legend_patches = [patches.Patch(color=color_map[t], label=t) for t in text_types]
    ax.legend(handles=legend_patches, loc='upper right')
    
    plt.tight_layout()
    plt.show()


In [ ]:
# Visualize top detection image
top_image_path = detections_per_image.index[0]
top_image_detections = df[df['image_path'] == top_image_path]
visualize_detections(top_image_path, top_image_detections)


## 5. Text Type Distribution


In [ ]:
# Distribution of text types
text_type_counts = df['text_type'].value_counts()
print("=== TEXT TYPE DISTRIBUTION ===")
print(text_type_counts)

fig, ax = plt.subplots(figsize=(10, 6))
text_type_counts.plot(kind='bar', ax=ax, edgecolor='black')
ax.set_xlabel('Text Type')
ax.set_ylabel('Count')
ax.set_title('Distribution of Detected Text Types')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 6. Sample Detected Texts


In [ ]:
# Sample of detected texts by type
for text_type in df['text_type'].unique():
    samples = df[df['text_type'] == text_type]['text'].dropna().head(10).tolist()
    print(f"\n=== {text_type.upper() if text_type else 'UNKNOWN'} ===")
    for s in samples:
        print(f"  • {s}")


## 7. Tiling Verification


In [ ]:
# Check tiling info
print("=== TILING CONFIGURATION ===")
print(f"Tiles per image (X): {df['n_tiles_x'].iloc[0]}")
print(f"Tiles per image (Y): {df['n_tiles_y'].iloc[0]}")
print(f"Total tiles per image: {df['n_tiles_x'].iloc[0] * df['n_tiles_y'].iloc[0]}")
print(f"\nGlobal bbox range:")
print(f"  Max X: {df['bbox_max_x'].iloc[0]}")
print(f"  Max Y: {df['bbox_max_y'].iloc[0]}")


In [ ]:
# Check bbox distribution across global coordinate space
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# X coordinate distribution
axes[0].hist(df['bbox_x1'].values, bins=50, alpha=0.7, label='x1')
axes[0].hist(df['bbox_x2'].values, bins=50, alpha=0.7, label='x2')
axes[0].set_xlabel('X Coordinate')
axes[0].set_ylabel('Count')
axes[0].set_title('X Coordinate Distribution')
axes[0].legend()

# Y coordinate distribution
axes[1].hist(df['bbox_y1'].values, bins=50, alpha=0.7, label='y1')
axes[1].hist(df['bbox_y2'].values, bins=50, alpha=0.7, label='y2')
axes[1].set_xlabel('Y Coordinate')
axes[1].set_ylabel('Count')
axes[1].set_title('Y Coordinate Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()


## 8. Summary


In [ ]:
print("="*60)
print("OCR DETECTION VERIFICATION SUMMARY")
print("="*60)
print(f"\n✓ Total images processed: {len(detections_per_image):,}")
print(f"✓ Total text detections: {len(df):,}")
print(f"✓ Average detections per image: {detections_per_image.mean():.2f}")
print(f"✓ Max detections in single image: {detections_per_image.max()}")
print(f"✓ Images with >10 detections: {len(detections_per_image[detections_per_image > 10]):,}")
print(f"\n✓ CONFIRMATION: Multiple detections ARE being accumulated properly!")
print("="*60)
